# Milestone 1: data preparation

**Goal:** turn the downloaded chapters into a clean, uniform corpus that
milestone 2 can align. See `PLAN.md`, section 4, milestone 1.

For every chapter in the splits this produces:
- `audio.wav`: mono, at the config sample rate, loudness-normalised;
- `chapter.json`: verses (raw and cleaned), the full cleaned chapter text
  (with the spoken announcement, if any), and the detected pauses.

Everything is then packed into one archive on Google Drive.

**Runtime:** CPU is enough. Needs notebook 00 to have run (it writes
`raw/sources.json`) and the notes from it applied to `configs/data.yaml`.

**Gate to milestone 2:** every chapter prepared, nothing important in the
dropped-character list, no speaking-rate outliers left unexplained, and the
pause plots look right for five random chapters.

## Setup

In [ ]:
# Get the code, install it, and connect Google Drive.
import subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/nicholas-on/TTS.git"
BRANCH = "main"
CODE_DIR = Path("/content/TTS")

def github_token():
    """The repository is private, so cloning needs a GitHub token, stored as a
    Colab secret named GITHUB_TOKEN with "Notebook access" switched on.
    See the README, "Running the proof of concept on Google Colab"."""
    try:
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    except ImportError:
        return None  # not in Colab
    except Exception as e:  # secret missing, or notebook access not granted
        print(f"No GITHUB_TOKEN available ({type(e).__name__}).")
        return None

def git(*args):
    result = subprocess.run(["git", *args], capture_output=True, text=True)
    if result.returncode != 0:
        message = result.stderr.replace(token, "***") if token else result.stderr
        raise RuntimeError(
            f"git {args[0]} failed:\n{message}\n"
            "If this says 'Repository not found' or 'could not read Username', the token is "
            "missing or wrong: add a Colab secret named GITHUB_TOKEN (key icon, left sidebar), "
            "switch on 'Notebook access', and rerun this cell.")

token = github_token()
if not CODE_DIR.exists():
    url = REPO_URL.replace("https://", f"https://x-access-token:{token}@") if token else REPO_URL
    git("clone", "-q", "-b", BRANCH, url, str(CODE_DIR))
else:
    git("-C", str(CODE_DIR), "pull", "-q")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(CODE_DIR)], check=True)
if str(CODE_DIR / "src") not in sys.path:
    sys.path.insert(0, str(CODE_DIR / "src"))

from bibletts.colab import mount_drive, gpu_name, disk_free_gb
from bibletts.config import load_config

mount_drive()
cfg = load_config(CODE_DIR / "configs" / "data.yaml")
DRIVE = Path(cfg["paths"]["drive_root"])   # durable
LOCAL = Path(cfg["paths"]["local_root"])   # fast, wiped when the session ends
RAW = DRIVE / "raw"                        # downloaded originals
for d in (DRIVE, LOCAL, RAW):
    d.mkdir(parents=True, exist_ok=True)
print("GPU:", gpu_name(), "| free local disk:", round(disk_free_gb()), "GB")

In [ ]:
import json
from collections import Counter
from bibletts.corpus import parse_vpl, resolve_splits, chapter_id

sources = json.loads((RAW / "sources.json").read_text())
chapters_text = parse_vpl(sources["vpl"])
audio_paths = sources["audio"]
splits = resolve_splits(cfg["splits"])

todo = [k for k in splits if chapter_id(*k) in audio_paths and k in chapters_text]
skipped = [k for k in splits if k not in todo]
print(f"{len(todo)} chapters to prepare.")
print("Skipped (no audio or no text; rerun notebook 00 if unexpected):", skipped or "none")
print("Sample rate:", cfg["audio"]["sample_rate"], "| announcement:", cfg["text"]["chapter_announcement"])

## 1. Text cleaning

Check what the cleaner removes and look at some examples. Every character in
"dropped" disappears from the training text. If any of them is spoken aloud,
add a rule to `src/bibletts/text.py` before going on.

In [ ]:
import random
from bibletts.text import clean_text

alphabet, strip = cfg["text"]["alphabet"], cfg["text"]["strip_accents"]
dropped, inventory = Counter(), Counter()
for key in todo:
    for v, raw in chapters_text[key]:
        inventory.update(clean_text(raw, alphabet, strip, dropped))

print("Dropped characters:", dropped.most_common() or "none")
print("\nCharacters in the cleaned text, by frequency:")
print("  ", " ".join(f"{c!r}:{n}" for c, n in inventory.most_common()))

random.seed(0)
print("\nRandom verses, raw then cleaned:")
for key in random.sample(todo, 6):
    v, raw = random.choice(chapters_text[key])
    print(f"  {key[0]} {key[1]}:{v}  {raw}")
    print(f"  {'':>9}  {clean_text(raw, alphabet, strip)}")

In [ ]:
from bibletts.corpus import announcement_text

example = announcement_text(cfg["text"]["chapter_announcement"], *todo[0])
print("Announcement for", todo[0], "->", repr(example) if example else "none configured")
if example:
    print("Cleaned:", clean_text(example, alphabet, strip))

## 2. Prepare every chapter

Decode, resample, normalise loudness, save WAV, clean text, detect pauses.
Chapters already done are skipped, so rerunning after a disconnect continues
where it stopped. To redo everything after changing a setting, set
`OVERWRITE = True`.

**Where the files go:** the Colab local disk (`LOCAL`), because it is much
faster than Drive. Section 6 copies the result to Drive as one archive.

In [ ]:
from tqdm.auto import tqdm
from bibletts.corpus import prepare_chapter, write_manifest

OVERWRITE = False
dropped = Counter()
failed = {}
for key in tqdm(todo):
    try:
        prepare_chapter(*key, splits[key], audio_paths[chapter_id(*key)], chapters_text[key],
                        cfg, LOCAL, dropped=dropped, overwrite=OVERWRITE)
    except Exception as e:
        failed[key] = repr(e)

manifest = write_manifest(LOCAL)
print(f"Prepared {len(manifest)} chapters. Failed: {failed or 'none'}")

## 3. Summary

In [ ]:
import pandas as pd

df = pd.DataFrame(manifest)
df["minutes"] = df.duration_s / 60
df["chars_per_s"] = df.n_chars / df.duration_s
df["pauses_per_min"] = df.n_pauses / df.minutes

summary = df.groupby("split").agg(chapters=("id", "count"), hours=("minutes", lambda m: m.sum() / 60),
                                  verses=("n_verses", "sum"), chars_per_s=("chars_per_s", "mean"),
                                  pauses_per_min=("pauses_per_min", "mean"))
print(summary.round(2).to_string())

## 4. Speaking-rate outliers

Every chapter should be read at a similar rate (characters of text per second
of audio). A chapter far from the others usually means a problem: the wrong
audio file for that text, music or an intro taking up time, missing verses, or
a text that does not match the recording. Listen to every outlier before going
on. Milestone 2's first round depends on the rate being steady.

In [ ]:
mean, std = df.chars_per_s.mean(), df.chars_per_s.std()
df["rate_z"] = (df.chars_per_s - mean) / std
outliers = df[df.rate_z.abs() > 2.5].sort_values("rate_z")
print(f"Mean {mean:.1f} chars/s, std {std:.2f}.")
print("Outliers:" if len(outliers) else "No outliers.")
if len(outliers):
    print(outliers[["id", "split", "minutes", "chars_per_s", "rate_z"]].round(2).to_string(index=False))

import matplotlib.pyplot as plt
df.chars_per_s.hist(bins=40)
plt.xlabel("characters per second"); plt.ylabel("chapters"); plt.show()

## 5. Pauses

Milestone 2's first round cuts each chapter at pauses roughly every 20 to 30
seconds, so there need to be plenty of pauses, spread evenly.

Look at the plots: orange should cover the gaps between phrases and sentences,
and never the middle of a word. If pauses are missed, raise
`pauses.threshold_db` (for example -35). If there are pauses inside words,
lower it (-45) or raise `pauses.min_duration_s`. Then rerun section 2 with
`OVERWRITE = True`.

In [ ]:
import numpy as np
from IPython.display import Audio, display
from bibletts.corpus import load_chapter
from bibletts.audio import load_audio
from bibletts.viz import plot_pauses

sr = cfg["audio"]["sample_rate"]
gaps, lengths = [], []
for row in manifest:
    m = load_chapter(LOCAL, row["book"], row["chapter"])
    starts = [s for s, _ in m["pauses"]]
    gaps += list(np.diff(starts))
    lengths += [e - s for s, e in m["pauses"]]

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].hist(lengths, bins=60, range=(0, 3)); axes[0].set(title="pause length (s)")
axes[1].hist(gaps, bins=60, range=(0, 30)); axes[1].set(title="time between pause starts (s)")
plt.show()
print(f"Median pause {np.median(lengths):.2f} s; median gap between pauses {np.median(gaps):.1f} s; "
      f"longest stretch with no pause {max(gaps):.1f} s.")

In [ ]:
random.seed(1)
for row in random.sample(manifest, 5):
    m = load_chapter(LOCAL, row["book"], row["chapter"])
    wav = LOCAL / "chapters" / m["id"] / "audio.wav"
    audio = load_audio(wav, sr, 0, 45)
    plot_pauses(audio, sr, m["pauses"], 0, 45, title=f"{m['id']} ({m['split']}): first 45 s")
    plt.show()
    print(m["text"][:300], "...")
    display(Audio(audio, rate=sr))

## 6. Save to Google Drive

Packs the prepared corpus into one `.tar` file on Drive, with a copy of the
config used. Later notebooks start by unpacking it to the local disk:

```python
from bibletts.corpus import restore_archive
restore_archive(DRIVE / "prepared" / "chapters_v1.tar", LOCAL.parent)
```

If you change a setting and prepare again, bump the version number rather than
overwrite, so earlier results stay reproducible.

In [ ]:
from bibletts.corpus import make_archive
from bibletts.config import save_config

VERSION = "v1"
archive = make_archive(LOCAL, DRIVE / "prepared" / f"chapters_{VERSION}.tar")
save_config(cfg, DRIVE / "prepared" / f"chapters_{VERSION}_config.yaml")
print(f"Saved {archive} ({archive.stat().st_size / 1e9:.2f} GB)")

## 7. Gate checklist

- [ ] All chapters prepared, none failed
- [ ] Nothing spoken aloud is in the dropped-character list
- [ ] Every speaking-rate outlier listened to and explained or fixed
- [ ] Pause plots look right for five random chapters
- [ ] Archive and config saved to Drive
- [ ] Config changes committed to the repository